# Warehouse Right-Sizing & Query Health Notebook

Portable diagnostic notebook for a new Snowflake environment. Run top to bottom.
It surfaces:

1. **Long-running queries** and the likely root cause (spillage, poor pruning/clustering, join explosion, queueing, cold compile, etc.)
2. **Spillage** to local/remote storage (memory pressure → usually an upsize signal)
3. **Clustering / pruning problems** (scanning far more partitions than a query's filters should require)
4. **Likely cartesian products / join explosions** (heuristic — always confirm in Query Profile before acting)
5. **Over-provisioned queries** — small workloads burning credits on warehouses that are too large
6. **Under-provisioned warehouses** — sustained queuing that signals a need for a larger warehouse or more clusters
7. **Warehouse utilization & idle spend** — credits paid for warehouses sitting mostly idle
8. **A rolled-up recommendation summary**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database (or `ACCOUNTADMIN`), so you can query `SNOWFLAKE.ACCOUNT_USAGE.*`.
- `SNOWFLAKE.ACCOUNT_USAGE` has up to ~45 min–3 hr latency. For anything in the last hour, swap the equivalent `INFORMATION_SCHEMA` table functions in (noted inline where relevant).
- Nothing here is destructive — every cell is a `SELECT`. Nothing writes to the account.

### How to use it
Set the parameters in the next cell, then run each section. Every SQL cell is intentionally self-contained (re-declares its own filters) so you can jump around, comment sections out, or hand individual cells to someone else without breaking the rest of the notebook.


In [ ]:
-- ============================================================
-- PARAMETERS — adjust these once per environment, then run the notebook.
-- Snowsight SQL cells don't support session variables cleanly across cells
-- the way scripting does, so these are re-declared with SET and referenced
-- via IDENTIFIER()/$var where supported. If your Snowsight version doesn't
-- carry SET variables between cells, just hardcode the values directly in
-- each cell's WHERE clause (marked <<LOOKBACK_DAYS>> etc.) instead.
-- ============================================================
SET lookback_days = 14;          -- how far back to analyze
SET long_query_seconds = 60;     -- "long-running" threshold, in seconds
SET small_query_mb = 100;        -- below this scanned volume, a query is "small"
SET large_warehouse_sizes = 'LARGE,X-LARGE,2X-LARGE,3X-LARGE,4X-LARGE,5X-LARGE,6X-LARGE';
SET min_pruning_ratio = 0.7;     -- flag as poor pruning if partitions_scanned/partitions_total exceeds this
SET min_queue_seconds_per_day = 300; -- 5 min/day average queued time = provisioning problem

SELECT $lookback_days AS lookback_days,
       $long_query_seconds AS long_query_seconds,
       $small_query_mb AS small_query_mb,
       $large_warehouse_sizes AS large_warehouse_sizes,
       $min_pruning_ratio AS min_pruning_ratio,
       $min_queue_seconds_per_day AS min_queue_seconds_per_day;


## 1. Long-running queries, with likely cause

Pulls the slowest queries in the window and breaks total elapsed time down into its
components (compile / execution / queued-for-warehouse / queued-for-repair / spillage),
plus a plain-language `likely_cause` tag per row so you can triage quickly.


In [ ]:
-- Top long-running queries with a root-cause guess
WITH base AS (
    SELECT
        query_id,
        query_text,
        user_name,
        warehouse_name,
        warehouse_size,
        cluster_number,
        start_time,
        total_elapsed_time / 1000.0                         AS elapsed_sec,
        compilation_time / 1000.0                            AS compile_sec,
        execution_time / 1000.0                               AS exec_sec,
        queued_provisioning_time / 1000.0                     AS queued_provision_sec,
        queued_repair_time / 1000.0                           AS queued_repair_sec,
        queued_overload_time / 1000.0                         AS queued_overload_sec,
        transaction_blocked_time / 1000.0                     AS blocked_sec,
        bytes_scanned,
        bytes_spilled_to_local_storage,
        bytes_spilled_to_remote_storage,
        partitions_scanned,
        partitions_total,
        rows_produced,
        percentage_scanned_from_cache,
        query_type,
        warehouse_size IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($large_warehouse_sizes, ','))) AS is_large_wh
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND execution_status = 'SUCCESS'
      AND total_elapsed_time >= $long_query_seconds * 1000
)
SELECT
    query_id,
    LEFT(query_text, 200) AS query_text_preview,
    user_name,
    warehouse_name,
    warehouse_size,
    start_time,
    ROUND(elapsed_sec, 1)              AS elapsed_sec,
    ROUND(compile_sec, 1)              AS compile_sec,
    ROUND(exec_sec, 1)                 AS exec_sec,
    ROUND(queued_provision_sec, 1)     AS queued_provision_sec,
    ROUND(queued_overload_sec, 1)      AS queued_overload_sec,
    ROUND(bytes_spilled_to_local_storage / POWER(1024,3), 2)  AS local_spill_gb,
    ROUND(bytes_spilled_to_remote_storage / POWER(1024,3), 2) AS remote_spill_gb,
    partitions_scanned,
    partitions_total,
    ROUND(DIV0(partitions_scanned, partitions_total), 3) AS pruning_ratio,
    rows_produced,
    ROUND(bytes_scanned / POWER(1024,3), 2) AS gb_scanned,
    CASE
        WHEN bytes_spilled_to_remote_storage > 0 THEN 'REMOTE SPILLAGE — warehouse undersized on memory, upsize or reduce working set'
        WHEN bytes_spilled_to_local_storage  > 0 THEN 'LOCAL SPILLAGE — memory pressure, consider upsizing or tuning query'
        WHEN queued_provision_sec > 5 OR queued_overload_sec > 5 THEN 'QUEUED — warehouse was provisioning/overloaded, consider bigger warehouse or more clusters'
        WHEN DIV0(partitions_scanned, partitions_total) > $min_pruning_ratio AND partitions_total > 100 THEN 'POOR PRUNING — likely clustering/predicate issue, scanning most of the table'
        WHEN compile_sec > (0.3 * elapsed_sec) AND compile_sec > 2 THEN 'HIGH COMPILE TIME — complex query plan or cold metadata cache'
        WHEN blocked_sec > 2 THEN 'TRANSACTION BLOCKED — lock contention, not a warehouse sizing issue'
        WHEN is_large_wh AND bytes_scanned < ($small_query_mb * POWER(1024,2)) THEN 'SMALL WORKLOAD ON LARGE WAREHOUSE — right-sizing candidate, see Section 5'
        ELSE 'COMPUTE-BOUND — genuinely large scan/aggregation, review query logic'
    END AS likely_cause
FROM base
ORDER BY elapsed_sec DESC
LIMIT 200;


**Reading this table:** `likely_cause` is a first-pass triage, not a verdict — for anything
flagged, open that `query_id` in Query Profile to confirm before changing warehouse config.


## 2. Spillage detail (memory pressure)

Spillage means the warehouse ran out of memory for an operation (usually a sort, join, or
aggregation) and spilled intermediate results to local disk or — worse — remote storage.
Remote spillage is the strongest "this warehouse is too small" signal in `QUERY_HISTORY`.


In [ ]:
-- Queries and warehouses most affected by spillage
SELECT
    warehouse_name,
    warehouse_size,
    COUNT(*)                                                   AS spilling_queries,
    ROUND(SUM(bytes_spilled_to_local_storage) / POWER(1024,3), 1)  AS total_local_spill_gb,
    ROUND(SUM(bytes_spilled_to_remote_storage) / POWER(1024,3), 1) AS total_remote_spill_gb,
    ROUND(AVG(total_elapsed_time) / 1000.0, 1)                 AS avg_elapsed_sec,
    ROUND(SUM(CASE WHEN bytes_spilled_to_remote_storage > 0 THEN 1 ELSE 0 END)
          / NULLIF(COUNT(*),0) * 100, 1)                       AS pct_with_remote_spill
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND (bytes_spilled_to_local_storage > 0 OR bytes_spilled_to_remote_storage > 0)
GROUP BY warehouse_name, warehouse_size
ORDER BY total_remote_spill_gb DESC, total_local_spill_gb DESC;


In [ ]:
-- Worst individual offenders, for drilling into Query Profile
SELECT
    query_id,
    LEFT(query_text, 200) AS query_text_preview,
    user_name,
    warehouse_name,
    warehouse_size,
    start_time,
    ROUND(bytes_spilled_to_local_storage / POWER(1024,3), 2)  AS local_spill_gb,
    ROUND(bytes_spilled_to_remote_storage / POWER(1024,3), 2) AS remote_spill_gb,
    ROUND(total_elapsed_time / 1000.0, 1)                      AS elapsed_sec,
    ROUND(bytes_scanned / POWER(1024,3), 2)                    AS gb_scanned
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND (bytes_spilled_to_local_storage > 0 OR bytes_spilled_to_remote_storage > 0)
ORDER BY bytes_spilled_to_remote_storage DESC, bytes_spilled_to_local_storage DESC
LIMIT 100;


## 3. Clustering / pruning problems

If a query's filters should let Snowflake skip most micro-partitions but it scans nearly
all of them anyway, that's a pruning problem — usually a missing or ineffective clustering
key, a predicate that can't prune (e.g. a function wrapped around the clustered column), or
a naturally unsorted load pattern. This also checks automatic clustering spend, since a
table can be burning credits on reclustering that isn't actually improving pruning.


In [ ]:
-- Queries with poor partition pruning on non-trivial tables
SELECT
    query_id,
    LEFT(query_text, 200) AS query_text_preview,
    warehouse_name,
    warehouse_size,
    start_time,
    partitions_scanned,
    partitions_total,
    ROUND(DIV0(partitions_scanned, partitions_total), 3) AS pruning_ratio,
    ROUND(bytes_scanned / POWER(1024,3), 2) AS gb_scanned,
    ROUND(total_elapsed_time / 1000.0, 1)   AS elapsed_sec
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND partitions_total > 100                       -- ignore trivially small tables
  AND DIV0(partitions_scanned, partitions_total) > $min_pruning_ratio
  AND query_type = 'SELECT'
ORDER BY gb_scanned DESC
LIMIT 200;


In [ ]:
-- Tables spending the most on automatic clustering — verify it's earning its keep
SELECT
    table_name,
    ROUND(SUM(credits_used), 2)              AS total_credits_used,
    ROUND(SUM(num_bytes_reclustered) / POWER(1024,3), 1) AS gb_reclustered,
    SUM(num_rows_reclustered)                 AS rows_reclustered,
    COUNT(*)                                   AS reclustering_events
FROM snowflake.account_usage.automatic_clustering_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
GROUP BY table_name
ORDER BY total_credits_used DESC
LIMIT 50;


If a table shows up in both results above — high reclustering spend *and* queries against
it still show poor pruning — the clustering key likely doesn't match the actual filter
patterns being used, and it's worth revisiting rather than just paying for more reclustering.


## 4. Likely cartesian products / join explosions (heuristic)

`ACCOUNT_USAGE` doesn't expose join types directly, so this section combines two signals:

- **Explicit `CROSS JOIN`** in the query text.
- **Row explosion**: queries producing an unusually large number of output rows relative
  to the data volume scanned (a real cartesian/fan-out join tends to output far more rows
  than the bytes scanned would suggest for a normal filter/join).

Treat every result here as a *candidate*, not a confirmed finding — open the query in
Query Profile and check the join operator's output-row multiplier against its inputs
before concluding anything.


In [ ]:
-- Candidate cartesian products / join explosions
WITH candidates AS (
    SELECT
        query_id,
        query_text,
        LEFT(query_text, 200) AS query_text_preview,
        warehouse_name,
        warehouse_size,
        start_time,
        total_elapsed_time / 1000.0 AS elapsed_sec,
        bytes_scanned,
        rows_produced,
        partitions_scanned,
        DIV0(rows_produced, NULLIF(bytes_scanned / POWER(1024,2), 0)) AS rows_per_mb_scanned,
        CONTAINS(UPPER(query_text), 'CROSS JOIN') AS has_explicit_cross_join
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND execution_status = 'SUCCESS'
      AND query_type = 'SELECT'
      AND rows_produced > 100000
)
SELECT
    query_id,
    query_text_preview,
    warehouse_name,
    warehouse_size,
    start_time,
    elapsed_sec,
    rows_produced,
    ROUND(bytes_scanned / POWER(1024,2), 1) AS mb_scanned,
    ROUND(rows_per_mb_scanned, 1)           AS rows_per_mb_scanned,
    has_explicit_cross_join,
    CASE
        WHEN has_explicit_cross_join THEN 'EXPLICIT CROSS JOIN'
        WHEN rows_per_mb_scanned > 5000 THEN 'ROW EXPLOSION — very high output-rows-per-MB-scanned, check join fan-out'
        ELSE 'REVIEW'
    END AS flag_reason
FROM candidates
WHERE has_explicit_cross_join OR rows_per_mb_scanned > 5000
ORDER BY rows_per_mb_scanned DESC
LIMIT 100;


## 5. Right-sizing: queries that should run on a smaller warehouse

Queries that scan very little data, don't spill, and run on a Large-or-bigger warehouse are
usually not benefiting from the extra size — you're paying for idle nodes on that query.
Grouped by warehouse and query tag/user so you can see whether it's a specific job, dashboard,
or user routing traffic to an oversized warehouse.


In [ ]:
-- Small workloads running on large warehouses
WITH flagged AS (
    SELECT
        query_id,
        user_name,
        query_tag,
        warehouse_name,
        warehouse_size,
        start_time,
        total_elapsed_time / 1000.0 AS elapsed_sec,
        bytes_scanned,
        bytes_spilled_to_local_storage,
        bytes_spilled_to_remote_storage,
        partitions_scanned
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND execution_status = 'SUCCESS'
      AND warehouse_size IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($large_warehouse_sizes, ',')))
      AND bytes_scanned < ($small_query_mb * POWER(1024,2))
      AND bytes_spilled_to_local_storage = 0
      AND bytes_spilled_to_remote_storage = 0
)
SELECT
    warehouse_name,
    warehouse_size,
    user_name,
    query_tag,
    COUNT(*)                                      AS small_query_count,
    ROUND(AVG(elapsed_sec), 2)                    AS avg_elapsed_sec,
    ROUND(AVG(bytes_scanned) / POWER(1024,2), 1)  AS avg_mb_scanned,
    MIN(start_time)                                AS first_seen,
    MAX(start_time)                                AS last_seen
FROM flagged
GROUP BY warehouse_name, warehouse_size, user_name, query_tag
ORDER BY small_query_count DESC
LIMIT 100;


If one `user_name` or `query_tag` dominates this list on a given warehouse, that's usually
the cleanest fix: route that job/user to its own small warehouse (or a lower-priority queue)
instead of resizing the shared one down and risking the few queries that *do* need the size.


## 6. Warehouses that may be under-provisioned (queuing)

Sustained queuing means requests are arriving faster than the warehouse can start them.
`WAREHOUSE_LOAD_HISTORY` gives 5-minute average load buckets — persistent `avg_queued_load`
or `avg_queued_provisioning` above zero, day after day, is the signal to either upsize the
warehouse or (on Enterprise+ edition) turn on / raise multi-cluster `MAX_CLUSTER_COUNT`.


In [ ]:
-- Daily average queuing per warehouse
SELECT
    warehouse_name,
    DATE_TRUNC('day', start_time) AS day,
    ROUND(AVG(avg_running), 2)             AS avg_running,
    ROUND(AVG(avg_queued_load), 2)         AS avg_queued_load,
    ROUND(AVG(avg_queued_provisioning), 2) AS avg_queued_provisioning,
    ROUND(AVG(avg_blocked), 2)             AS avg_blocked
FROM snowflake.account_usage.warehouse_load_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
GROUP BY warehouse_name, day
HAVING AVG(avg_queued_load) > 0 OR AVG(avg_queued_provisioning) > 0
ORDER BY day DESC, avg_queued_load DESC;


In [ ]:
-- Warehouses ranked by total queued time across the window (candidates for upsize / more clusters)
SELECT
    warehouse_name,
    warehouse_size,
    COUNT(*)                                             AS long_or_queued_queries,
    ROUND(SUM(queued_provisioning_time) / 1000.0, 1)      AS total_queued_provision_sec,
    ROUND(SUM(queued_overload_time) / 1000.0, 1)          AS total_queued_overload_sec,
    ROUND(AVG(queued_provisioning_time + queued_overload_time) / 1000.0, 2) AS avg_queue_sec_per_query
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND (queued_provisioning_time > 0 OR queued_overload_time > 0)
GROUP BY warehouse_name, warehouse_size
ORDER BY total_queued_overload_sec DESC, total_queued_provision_sec DESC
LIMIT 50;


`queued_overload_time` (query waited because the warehouse's current cluster(s) were
saturated) is the stronger signal for **more clusters**; `queued_provisioning_time` (waiting
for compute to spin up) is more often about **auto-suspend/auto-resume being too aggressive**
for a spiky workload rather than a sizing problem.


## 7. Warehouse utilization & idle spend

Credits are billed per-second while a warehouse is running, whether or not it's actually
executing a query. This section shows credit spend per warehouse and, separately, current
`AUTO_SUSPEND` settings — a warehouse with a lot of low-utilization runtime and a long
auto-suspend is usually leaking credits on idle time rather than a compute-sizing problem.


In [ ]:
-- Credit spend by warehouse
SELECT
    warehouse_name,
    ROUND(SUM(credits_used), 2)          AS total_credits,
    ROUND(SUM(credits_used_compute), 2)  AS compute_credits,
    ROUND(SUM(credits_used_cloud_services), 2) AS cloud_services_credits,
    COUNT(DISTINCT DATE_TRUNC('day', start_time)) AS active_days
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY total_credits DESC;


In [ ]:
-- Current warehouse configuration (size, auto-suspend, multi-cluster settings)
SHOW WAREHOUSES;


In [ ]:
-- Parse the SHOW WAREHOUSES output above
SELECT
    "name"                    AS warehouse_name,
    "size",
    "min_cluster_count",
    "max_cluster_count",
    "auto_suspend"            AS auto_suspend_seconds,
    "auto_resume",
    "started_clusters",
    "running",
    "queued"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY warehouse_name;


Cross-reference this with Section 6: a warehouse with `max_cluster_count = 1` that shows up
heavily in the queuing results is a straightforward multi-cluster candidate (Enterprise
edition or above). A warehouse with a high `auto_suspend` value and low query density is an
idle-spend candidate — lowering `AUTO_SUSPEND` (e.g. to 60s) typically has no correctness
impact and directly cuts idle billing.


## 8. Rolled-up recommendation summary

One row per warehouse, pulling the signals above together into a single "what to look at
first" view.


In [ ]:
-- Summary rollup per warehouse
WITH credits AS (
    SELECT warehouse_name, SUM(credits_used) AS total_credits
    FROM snowflake.account_usage.warehouse_metering_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
    GROUP BY warehouse_name
),
spill AS (
    SELECT warehouse_name,
           SUM(bytes_spilled_to_local_storage)  AS local_spill_bytes,
           SUM(bytes_spilled_to_remote_storage) AS remote_spill_bytes
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
    GROUP BY warehouse_name
),
queueing AS (
    SELECT warehouse_name,
           SUM(queued_provisioning_time) AS queued_provision_ms,
           SUM(queued_overload_time)     AS queued_overload_ms
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
    GROUP BY warehouse_name
),
smallq AS (
    SELECT warehouse_name, COUNT(*) AS small_query_count
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND warehouse_size IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($large_warehouse_sizes, ',')))
      AND bytes_scanned < ($small_query_mb * POWER(1024,2))
      AND bytes_spilled_to_local_storage = 0
      AND bytes_spilled_to_remote_storage = 0
    GROUP BY warehouse_name
),
pruning AS (
    SELECT warehouse_name, COUNT(*) AS poor_pruning_query_count
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND partitions_total > 100
      AND DIV0(partitions_scanned, partitions_total) > $min_pruning_ratio
    GROUP BY warehouse_name
)
SELECT
    c.warehouse_name,
    ROUND(c.total_credits, 1)                              AS total_credits_used,
    ROUND(s.local_spill_bytes / POWER(1024,3), 1)          AS local_spill_gb,
    ROUND(s.remote_spill_bytes / POWER(1024,3), 1)         AS remote_spill_gb,
    ROUND(q.queued_provision_ms / 1000.0, 0)               AS total_queued_provision_sec,
    ROUND(q.queued_overload_ms / 1000.0, 0)                AS total_queued_overload_sec,
    COALESCE(sm.small_query_count, 0)                       AS small_queries_on_large_wh,
    COALESCE(p.poor_pruning_query_count, 0)                 AS poor_pruning_queries,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        IFF(s.remote_spill_bytes > 0, 'UPSIZE (remote spill)', NULL),
        IFF(s.local_spill_bytes > 0 AND s.remote_spill_bytes = 0, 'CONSIDER UPSIZE (local spill)', NULL),
        IFF(q.queued_overload_ms > $min_queue_seconds_per_day * 1000 * $lookback_days, 'ADD CLUSTERS (sustained overload queuing)', NULL),
        IFF(q.queued_provision_ms > $min_queue_seconds_per_day * 1000 * $lookback_days, 'REVIEW AUTO_SUSPEND/RESUME (provisioning queuing)', NULL),
        IFF(COALESCE(sm.small_query_count, 0) > 20, 'DOWNSIZE OR ROUTE ELSEWHERE (many small queries)', NULL),
        IFF(COALESCE(p.poor_pruning_query_count, 0) > 20, 'REVIEW CLUSTERING KEYS (poor pruning)', NULL)
    ), '; ') AS recommendations
FROM credits c
LEFT JOIN spill s     ON c.warehouse_name = s.warehouse_name
LEFT JOIN queueing q  ON c.warehouse_name = q.warehouse_name
LEFT JOIN smallq sm   ON c.warehouse_name = sm.warehouse_name
LEFT JOIN pruning p   ON c.warehouse_name = p.warehouse_name
ORDER BY total_credits_used DESC;


### Notes, caveats, and next steps

- **Thresholds are starting points, not rules.** `small_query_mb`, `min_pruning_ratio`, and
  `min_queue_seconds_per_day` in the parameters cell should be tuned to the environment —
  a healthtech workload with mostly small dimensional lookups will have a very different
  baseline than a workload doing large batch transforms.
- **Confirm before acting.** Every heuristic here (cartesian detection especially) is meant
  to narrow down where to look, not to hand you a final answer — open the flagged
  `query_id` in Query Profile before resizing anything.
- **Cost vs. latency tradeoff.** Upsizing fixes spillage and queuing fast but costs more per
  second; adding clusters costs more only while they're actually running. Downsizing/routing
  small queries elsewhere is usually the highest-leverage, lowest-risk change to make first.
- **Latency of ACCOUNT_USAGE.** If you need up-to-the-minute data (e.g. investigating an
  incident happening right now), use `TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(...))` and
  `TABLE(INFORMATION_SCHEMA.WAREHOUSE_LOAD_HISTORY(...))` instead — same columns, no
  latency, but capped to the last 7 days and requires the query to run in the relevant
  database context.
- **Resource Monitors** aren't covered here (this notebook is query/warehouse behavior, not
  spend caps) — pair this analysis with `SHOW RESOURCE MONITORS;` if the org also wants
  hard spend guardrails while right-sizing is in progress.
